# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [2]:
# Core stack — MuJoCo is Apache 2.0 since the DeepMind acquisition, pip-installable
%uv pip install mujoco gymnasium[mujoco,other] imageio[ffmpeg]

# LeRobot (Hugging Face) — policies, datasets, sim env wrappers
%uv pip install lerobot[smolvla]

# Sim envs LeRobot policies were actually trained on
%uv pip install gym-pusht gym-aloha

%uv pip install num2words "pymunk>=6.6.0,<7.0.0"

Using Python 3.12.6 environment at: /usr/local
Resolved 36 packages in 479ms
⠙ Preparing packages... (0/10)
⠙ Preparing packages... (0/10)
⠙ Preparing packages... (0/10)
proglog    ------------------------------     0 B/6.19 KiB
⠙ Preparing packages... (0/10)
proglog    ------------------------------ 6.19 KiB/6.19 KiB
⠙ Preparing packages... (0/10)
proglog    ------------------------------ 6.19 KiB/6.19 KiB
cloudpickle ------------------------------ 14.78 KiB/21.71 KiB
⠙ Preparing packages... (0/10)
proglog    ------------------------------ 6.19 KiB/6.19 KiB
cloudpickle ------------------------------ 14.78 KiB/21.71 KiB
moviepy    ------------------------------     0 B/126.83 KiB
⠙ Preparing packages... (0/10)
proglog    ------------------------------ 6.19 KiB/6.19 KiB
cloudpickle ------------------------------ 14.78 KiB/21.71 KiB
moviepy    ------------------------------     0 B/126.83 KiB
gymnasium  ------------------------------     0 B/931.55 KiB
⠙ Preparing packages... (0/10)
prog

In [3]:
%uv pip install -U transformers

Using Python 3.12.6 environment at: /usr/local
Resolved 27 packages in 334ms
⠙ Preparing packages... (0/17)
⠙ Preparing packages... (0/17)
typing-extensions ------------------------------     0 B/44.50 KiB
⠙ Preparing packages... (0/17)
annotated-doc ------------------------------     0 B/5.18 KiB
typing-extensions ------------------------------     0 B/44.50 KiB
⠙ Preparing packages... (0/17)
annotated-doc ------------------------------     0 B/5.18 KiB
typing-extensions ------------------------------     0 B/44.50 KiB
⠙ Preparing packages... (0/17)
annotated-doc ------------------------------     0 B/5.18 KiB
typing-extensions ------------------------------ 14.74 KiB/44.50 KiB
⠙ Preparing packages... (0/17)
annotated-doc ------------------------------ 5.18 KiB/5.18 KiB
typing-extensions ------------------------------ 14.74 KiB/44.50 KiB
⠙ Preparing packages... (0/17)
annotated-doc ------------------------------ 5.18 KiB/5.18 KiB
typing-extensions ------------------------------ 14.74 

In [4]:
!apt-get update && apt install -y libosmesa6-dev libgl1-mesa-glx xvfb x11-utils

Get:1 http://deb.debian.org/debian bookworm InRelease [151 kB]
Get:2 http://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:3 http://deb.debian.org/debian-security bookworm-security InRelease [34.8 kB]
Get:4 http://deb.debian.org/debian bookworm/main amd64 Packages [8790 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  InRelease [1578 B]
Get:6 http://deb.debian.org/debian-security bookworm-security/main amd64 Packages [335 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  Packages [2211 kB]
Fetched 11.6 MB in 4s (2641 kB/s)

N: Repository 'http://deb.debian.org/debian bookworm InRelease' changed its 'Version' value from '12.11' to '12.15'



The following additional packages will be installed:
  libdrm-dev libfontenc1 libgbm1 libgl-dev libgl1-mesa-dri libglapi-mesa libglx-dev libglx-mesa0 libice6
  libosmesa6 libpciaccess-dev libpthread-stubs0-dev libsm6 libunwind8 libx11-dev libxau-dev libxaw7 libxcb1-

In [4]:
!Xvfb :1 &

UsageError: Line magic function `%Xvfb` not found.


In [10]:
!ps -ef | grep Xvfb

root      1194   549  0 23:09 ?        00:00:00 [Xvfb] <defunct>
root      1208   549 66 23:11 ?        00:00:00 /usr/bin/sh -c ps -ef | grep Xvfb
root      1210  1208  0 23:11 ?        00:00:00 grep Xvfb


In [2]:
!rm /tmp/.X1-lock

In [1]:
!xdpyinfo -display :1

/usr/bin/sh: 1: xdpyinfo: not found


In [1]:
import subprocess

startX11 = """
Xvfb :1 &

display_ready=0
tries=0

while [ $display_ready = 1 ] || [ $tries -lt 10 ]
do
    if xdpyinfo -display :1 >& /dev/null ; then break
    else echo "Virutal display not ready yet (try $tries)" && sleep 1 ; fi
    tries=`expr $tries + 1`
done

if xdpyinfo -display :1 >& /dev/null ; then echo "Display exists"
else echo "Display invalid, too many tries" && exit -1 ; fi

export DISPLAY=:1
echo $DISPLAY
"""

x11 = subprocess.run(startX11, shell=True, text=True, check=True, executable="/bin/bash")

Virutal display not ready yet (try 0)
Display exists
:1


In [2]:
x11

CompletedProcess(args='Xvfb :1 &', returncode=0)

In [2]:
import mujoco, numpy as np, imageio

import os

os.environ["DISPLAY"] = ":1"

model = mujoco.MjModel.from_xml_string("""
<mujoco>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="1 1 .1"/>
    <body pos="0 0 .5">
      <joint type="free"/>
      <geom type="box" size=".1 .1 .1" rgba="0.8 0.2 0.2 1"/>
    </body>
  </worldbody>
</mujoco>
""")
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

frames = []
for i in range(50):
    print(i)
    mujoco.mj_step(model, data)          # physics
    renderer.update_scene(data)          # sync camera
    frames.append(renderer.render())     # HxWx3 numpy

imageio.mimsave("drop.mp4", frames, fps=int(round(1/model.opt.timestep)))


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


In [4]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

import os
os.environ["DISPLAY"] = ":1"

env = gym.make("HalfCheetah-v5", render_mode="rgb_array")
env = RecordVideo(env, video_folder="videos", episode_trigger=lambda ep: True,
                  name_prefix="random-policy")

obs, info = env.reset(seed=42)
for step in range(100):
    print(step)
    action = env.action_space.sample()   # replace with policy output later
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        obs, info = env.reset()

env.close()  # REQUIRED — video is flushed on close


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [4]:
%uv pip install -U transformers

Using Python 3.12.6 environment at: /usr/local
Resolved 27 packages in 176ms
⠙ Preparing packages... (0/19)
⠙ Preparing packages... (0/19)
⠙ Preparing packages... (0/19)
packaging  ------------------------------     0 B/126.91 KiB
⠙ Preparing packages... (0/19)
packaging  ------------------------------ 14.76 KiB/126.91 KiB
⠙ Preparing packages... (0/19)
anyio      ------------------------------     0 B/122.86 KiB
packaging  ------------------------------ 14.76 KiB/126.91 KiB
⠙ Preparing packages... (0/19)
anyio      ------------------------------ 14.77 KiB/122.86 KiB
packaging  ------------------------------ 14.76 KiB/126.91 KiB
⠙ Preparing packages... (0/19)
typer      ------------------------------     0 B/119.99 KiB
anyio      ------------------------------ 14.77 KiB/122.86 KiB
packaging  ------------------------------ 14.76 KiB/126.91 KiB
⠙ Preparing packages... (0/19)
filelock   ------------------------------     0 B/97.52 KiB
typer      ------------------------------     0 B/119.

In [3]:
%uv pip install num2words

Using Python 3.12.6 environment at: /usr/local
Resolved 2 packages in 981ms
Building docopt==0.6.2
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
⠙ Preparing packages... (0/2)
Building docopt==0.6.2
Building docopt==0.6.2
Building docopt==0.6.2
Building docopt==0.6.2
Building docopt==0.6.2
   Built docopt==0.6.2
Prepared 2 packages in 907ms
Installed 2 packages in 3ms
 + docopt==0.6.2
 + num2words==0.5.14
Note: you may need to restart the kernel to use updated packages.


In [3]:
%uv pip install "pymunk>=6.6.0,<7.0.0"

Using Python 3.12.6 environment at: /usr/local
Resolved 3 packages in 134ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 34ms
Uninstalled 1 package in 1ms
Installed 1 package in 34ms
 - pymunk==7.3.0
 + pymunk==6.11.1
Note: you may need to restart the kernel to use updated packages.


In [10]:
%uv pip install lerobot[smolvla]

Using Python 3.12.6 environment at: /usr/local
Resolved 71 packages in 186ms
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
⠙ Preparing packages... (0/4)
packaging  ------------------------------ 16.00 KiB/64.91 KiB
⠙ Preparing packages... (0/4)
packaging  ------------------------------ 16.00 KiB/64.91 KiB
⠙ Preparing packages... (0/4)
packaging  ------------------------------ 16.00 KiB/64.91 KiB
accelerate ------------------------------ 14.81 KiB/380.12 KiB
⠙ Preparing packages... (0/4)
packaging  ------------------------------ 16.00 KiB/64.91 KiB
accelerate ------------------------------ 14.81 KiB/380.12 KiB
⠙ Preparing packages... (0/4)
packaging  ------------------------------ 32.00 KiB/64.91 KiB
accelerate ------------------------------ 14.81 KiB/380.12 KiB
⠙ Preparing packages... (0/4)
packaging  ------------------------------ 32.00 KiB/64.91 KiB
accelerate ------------------------------ 30.81 KiB/380.12 KiB
⠙ Preparing packages... (0/4)
packaging  ------------------

In [2]:
import gymnasium as gym, torch, numpy as np
from gymnasium.wrappers import RecordVideo
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

import gym_pusht

import os
os.environ["DISPLAY"] = ":1"

# --- load pretrained policy from the HF Hub ---

# --- Load a Push-T specific model (Action Dim = 2) ---
# Instead of the generic base model (Action Dim = 6)
model_name = "crislmfroes/smolvla-pusht"
policy = SmolVLAPolicy.from_pretrained(model_name)
policy.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
policy.to(device)

# 2. Instantiate the pre- and post-processors.
# This pipeline handles: Image/State normalization, Text Tokenization, Batching, and Device placement.
preprocess, postprocess = make_pre_post_processors(
    policy_cfg=policy.config,
    pretrained_path=model_name,
    # Override the device to match your environment (e.g. if training on GPU but eval on CPU/MPS)
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

# --- env: PushT is the "hello world" of VLAs ---
env = gym.make("gym_pusht/PushT-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")
env = RecordVideo(env, video_folder="videos", episode_trigger=lambda e: True)


[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/site-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/site-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

...

policy_preprocessor_step_5_normalizer_pr(…): downloading bytes:           |  0.00B            

policy_postprocessor.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

policy_postprocessor_step_0_unnormalizer(…): reconstructing file:   0%|          |  0.00B / 4.20kB            

policy_postprocessor_step_0_unnormalizer(…): downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [11]:

obs, info = env.reset(seed=0)
policy.reset()  # clears internal action queue — important!

episode_return = 0.0
for step in range(300):
    # --- observation batch, matching the policy's training keys ---
    #print(obs)
    raw_batch = {
        "observation.image":  torch.from_numpy(obs["pixels"])
                .permute(2, 0, 1)      # (H, W, C) → (C, H, W)
                .float().div(255.0)    # uint8 [0,255] → float [0,1]
                .unsqueeze(0)          # (C, H, W) → (1, C, H, W)
                .to(device),       # Raw H,W,3 uint8 array, converted
        "observation.state":        torch.from_numpy(obs["agent_pos"]).float().unsqueeze(0).to(device), # Raw float array
        "task": ["push the T-shaped block onto the target area marked by green color"],  # language conditioning
    }
    # 4. Pass through the preprocessor
    # This adds newlines to text, tokenizes it, normalizes images/states, 
    # adds the batch dimension, and moves tensors to the correct device.
    batch = preprocess(raw_batch)

    with torch.inference_mode():
        action = policy.select_action(batch)          # (action_dim,) tensor
    
    # 6. Postprocess the action (handles unnormalization and CPU transfer)
    action = postprocess(action)
    action = action.squeeze(0).cpu().numpy()

    obs, reward, terminated, truncated, info = env.step(action)
    episode_return += reward
    if terminated or truncated:
        break

env.close()
print(f"return: {episode_return:.2f}")


return: 4.54


In [9]:
!cat -n /usr/local/lib/python3.12/site-packages/lerobot/policies/smolvla/modeling_smolvla.py

     1	#!/usr/bin/env python
     2	
     3	# Copyright 2025 HuggingFace Inc. team. All rights reserved.
     4	#
     5	# Licensed under the Apache License, Version 2.0 (the "License");
     6	# you may not use this file except in compliance with the License.
     7	# You may obtain a copy of the License at
     8	#
     9	#     http://www.apache.org/licenses/LICENSE-2.0
    10	#
    11	# Unless required by applicable law or agreed to in writing, software
    12	# distributed under the License is distributed on an "AS IS" BASIS,
    13	# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
    14	# See the License for the specific language governing permissions and
    15	# limitations under the License.
    16	
    17	"""
    18	SmolVLA:
    19	
    20	[Paper](https://huggingface.co/papers/2506.01844)
    21	
    22	Designed by Hugging Face.
    23	
    24	Install smolvla extra dependencies:
    25	```bash
    26	pip install -e ".[smolvla]"
    27	```
    28	
    

In [12]:
!ls  -la videos

total 333
drwxr-xr-x 1 root root    140 Aug 26 01:22 .
drwx------ 1 root root    120 Aug 26 01:10 ..
-rw-r--r-- 1 root root   4038 Aug 26 01:11 rl-video-episode-0.mp4
-rw-r--r-- 1 root root   4038 Aug 26 01:13 rl-video-episode-1.mp4
-rw-r--r-- 1 root root   4038 Aug 26 01:13 rl-video-episode-2.mp4
-rw-r--r-- 1 root root 175571 Aug 26 01:13 rl-video-episode-3.mp4
-rw-r--r-- 1 root root 152491 Aug 26 01:22 rl-video-episode-4.mp4


In [13]:
!curl -X POST https://tempfile.org/api/upload/local \
  -F "files=@videos/rl-video-episode-4.mp4" \
  -F "expiryHours=24"

{"success":true,"files":[{"id":"RmNZQ7PFh9P","name":"rl-video-episode-4.mp4","size":152491,"url":"https://tempfile.org/RmNZQ7PFh9P/","expiryTime":1787793767776}],"message":"1 file(s) uploaded successfully"}